<a href="https://colab.research.google.com/github/2BerbyMarty2/GeminiDoc-Insights/blob/main/Gemini_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U youtube-transcript-api google-genai

In [ ]:
!pip install -q langchain-text-splitters

In [ ]:
from google.colab import userdata
key = userdata.get('GEMINI_API_KEY')

In [ ]:
from google import genai
client = genai.Client(api_key = key)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# No location parameter needed here!
response = client.models.generate_content(
    model="gemini-2.5-flash",  # 1.5-flash has the most reliable free quota
    contents="Hello!"
)
print(response.text)

Hello there! How can I help you today?


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi

def get_transcript_text(video_url):
    video_id = video_url.split("v=")[1].split("&")[0]

    try:
        # 1. Initialize API
        ytt_api = YouTubeTranscriptApi()

        # 2. Fetch the transcript (returns a list of FetchedTranscriptSnippet objects)
        transcript_snippets = ytt_api.fetch(video_id)

        # 3. Access 'text' attribute using dot notation
        full_transcript = " ".join([snippet.text for snippet in transcript_snippets])

        return full_transcript

    except Exception as e:
        return f"Error: {str(e)}"

# Example Usage:
video_full_transcript = get_transcript_text("https://www.youtube.com/watch?v=eMlx5fFNoYc&list=TLPQMDIwNTIwMjYKFue5j5rZ5A&index=4")

In [ ]:
# Assuming 'client' is already initialized as shown before
model_id = "gemini-2.5-flash"

response = client.models.count_tokens(
    model=model_id,
    contents=video_full_transcript
)

print(f"Total Tokens: {response.total_tokens}")

Total Tokens: 5682


In [ ]:
pdf_path = "/content/drive/MyDrive/Machine Learning/2026MAY/data/attention_is_all_you_need.pdf"

In [ ]:
import time

# 1. Upload the PDF to Gemini's File API
file_data = client.files.upload(file=pdf_path)

# 2. Wait for processing (important for large PDFs)
while file_data.state == "PROCESSING":
    print(".", end="")
    time.sleep(2)
    file_data = client.files.get(name=file_data.name)

In [ ]:
# 3. Ask Gemini to extract specific data
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[file_data, "Extract all main topics into a JSON table."]
)

print(response.text)

```json
[
  {
    "topic": "Title",
    "details": "Attention Is All You Need"
  },
  {
    "topic": "Authors",
    "details": [
      {"name": "Ashish Vaswani", "affiliation": "Google Brain"},
      {"name": "Noam Shazeer", "affiliation": "Google Brain"},
      {"name": "Niki Parmar", "affiliation": "Google Research"},
      {"name": "Llion Jones", "affiliation": "Google Research"},
      {"name": "Aidan N. Gomez", "affiliation": "University of Toronto (Work performed while at Google Brain)"},
      {"name": "Illia Polosukhin", "affiliation": "Independent (Work performed while at Google Research)"},
      {"name": "Jakob Uszkoreit", "affiliation": "Google Research"},
      {"name": "Łukasz Kaiser", "affiliation": "Google Brain"}
    ]
  },
  {
    "topic": "Abstract Summary",
    "details": "Introduces Transformer, a novel network architecture relying solely on attention mechanisms, eschewing recurrence and convolutions. It achieves state-of-the-art results on machine translation task

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Define the splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,    # Target size (roughly 200-300 words)
    chunk_overlap=100,  # 10% overlap so context isn't lost at the edges
    separators=["\n\n", "\n", ".", " ", ""] # Order of priority for splitting
)

# 2. Create the chunks
chunks = text_splitter.split_text(video_full_transcript)

# 3. View the results
print(f"Original length: {len(video_full_transcript)} characters")
print(f"Number of chunks created: {len(chunks)}")

# Print a sample chunk
print("\n--- Sample Chunk 1 ---")
print(chunks[0])

Original length: 27820 characters
Number of chunks created: 32

--- Sample Chunk 1 ---
In the last chapter, you and I started to step through the internal workings of a transformer. This is one of the key pieces of technology inside large language models, and a lot of other tools in the modern wave of AI. It first hit the scene in a now-famous 2017 paper called Attention is All You Need, and in this chapter you and I will dig into what this attention mechanism is, visualizing how it processes data. As a quick recap, here's the important context I want you to have in mind. The goal of the model that you and I are studying is to take in a piece of text and predict what word comes next. The input text is broken up into little pieces that we call tokens, and these are very often words or pieces of words, but just to make the examples in this video easier for you and me to think about, let's simplify by pretending that tokens are always just words. The first step in a transformer is to asso